In [12]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()

# 모델 선언
model = init_chat_model("gpt-4o-mini")

In [13]:
from langchain.agents.middleware import after_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage, HumanMessage

# 평가용 판사(Judge) 모델 초기화
safety_model = init_chat_model("gpt-5-mini")

@after_agent
def answer_leakage_guardrail(state, runtime) :
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3. 결과에 따른 개입 (Intervention)
    if "LEAKED" in result.content:

        original_user_question = state["messages"][0].content # 첫 번째 메시지(사용자 질문) 가져오기

        print(f"원래 사용자의 질문: {original_user_question}")

        correction_prompt = f"""
        당신은 친절한 튜터입니다.
        정답을 말하지 말고, 정답을 찾아갈 수 있도록 안내해주세요.
        사용자 질문: {original_user_question}
        문제가 되었던 이전 답변: {last_message.content} (이 답변은 정답을 너무 직접적으로 말했으니 수정이 필요합니다.)
        """

        print('last_message', last_message.content)

        # LLM을 다시 호출하여 교정된 답변을 생성 (1회 더 호출하기 때문에 비용 발생하지만 품질 확보)
        correction_result = model.invoke([HumanMessage(content=correction_prompt)])


        # 원래의 유출된 답변을 교정된 답변으로 덮어씌움
        last_message.content = correction_result.content

    return None


In [14]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[answer_leakage_guardrail],
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 이 문제 너무 어려워. 그냥 정답 알려줘."}]
})

result['messages'][-1].content

원래 사용자의 질문: 직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 이 문제 너무 어려워. 그냥 정답 알려줘.
last_message 5

설명: 피타고라스의 정리에 따라 빗변 c satisfy c^2 = 3^2 + 4^2 = 9 + 16 = 25이므로, c = √25 = 5입니다.


'그 문제를 함께 풀어볼까요? 직각 삼각형에서 빗변의 길이를 구하기 위해 피타고라스의 정리를 사용할 수 있습니다. 피타고라스의 정리는 다음과 같아요:\n\n\\[ c^2 = a^2 + b^2 \\]\n\n여기서 \\( c \\)는 빗변의 길이, \\( a \\)와 \\( b \\)는 직각변의 길이입니다. \n\n주어진 직각변의 길이는 각각 3과 4이라고 했는데, 이를 식에 대입해보면 어떤 식을 얻을 수 있을까요? 먼저 \\( 3 \\)과 \\( 4 \\)를 각각 \\( a \\)와 \\( b \\)로 구해서 풀어볼까요? 그러면 \\( c^2 = 3^2 + 4^2 \\)가 되겠죠. 다음 단계로 가볼까요? 여기서 \\( c^2 \\)의 값을 계산해보세요. 어떤 숫자가 나오나요?'